In [149]:
from sklearn import model_selection, compose , preprocessing , impute, tree , linear_model , ensemble , metrics , neighbors, svm
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor

In [104]:
# Importing Data & Top 5 Rows
df = pd.read_csv("Smart_Outcome_Predictor_Dataset_5200.csv")
df.head()

,student_id,age,country_region,device_type,education_background,course_level,course_category,course_start_date,week_of_year,sessions,time_spent_hours,videos_watched,quiz_attempts,assignments_submitted,forum_posts,avg_quiz_score,attendance_rate,completion_status,final_score
0,700001,32,Europe,Laptop,Undergrad,Intermediate,Business,2024-03-18,12,1,7.6,1,6,1,1,53.3,0.655,0,49.8
1,700002,17,Europe,Laptop,Undergrad,Intermediate,Programming,2024-08-22,34,16,27.2,6,4,7,1,51.5,1.000,1,84.0
2,700003,25,Europe,Mobile,Graduate,Advanced,Programming,2024-09-28,39,6,7.1,16,2,2,0,62.2,0.810,0,62.5
3,700004,26,Asia,Mobile,Undergrad,Beginner,Design,2024-03-09,10,34,22.1,57,9,6,0,59.3,0.875,1,89.5
4,700005,26,Asia,Tablet,WorkingPro,Advanced,Business,2024-03-21,12,22,32.3,41,9,2,0,65.1,0.814,0,67.4


In [105]:
#information of data 
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5200 entries, 0 to 5199
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             5200 non-null   int64  
 1   age                    5200 non-null   int64  
 2   country_region         5200 non-null   object 
 3   device_type            5200 non-null   object 
 4   education_background   5200 non-null   object 
 5   course_level           5200 non-null   object 
 6   course_category        5200 non-null   object 
 7   course_start_date      5200 non-null   object 
 8   week_of_year           5200 non-null   int64  
 9   sessions               5200 non-null   int64  
 10  time_spent_hours       5088 non-null   float64
 11  videos_watched         5200 non-null   int64  
 12  quiz_attempts          5200 non-null   int64  
 13  assignments_submitted  5200 non-null   int64  
 14  forum_posts            5200 non-null   int64  
 15  avg_

In [106]:
#Shape of data
print(df.shape)

(5200, 19)


In [107]:
#Null Values
print(df.isnull().sum())

student_id                 0
age                        0
country_region             0
device_type                0
education_background       0
course_level               0
course_category            0
course_start_date          0
week_of_year               0
sessions                   0
time_spent_hours         112
videos_watched             0
quiz_attempts              0
assignments_submitted      0
forum_posts                0
avg_quiz_score            81
attendance_rate           80
completion_status          0
final_score                0
dtype: int64


### Part B: Dataset Understanding & Preparation

In [108]:
classification_target = "completion_status"
regression_target = "final_score"

In [109]:
# Classification data
X_class = df.drop(columns=[classification_target, regression_target])
y_class = df[classification_target]

In [110]:
#identify numeric and categorical columns
numeric_columns = X_class.select_dtypes(include=np.number).columns
categorical_columns = X_class.select_dtypes(exclude=np.number).columns

In [111]:
# Regression data
X_reg = df.drop(columns=[classification_target, regression_target])
y_reg = df[regression_target]

In [112]:
for col in categorical_columns:
    X_class[col] = preprocessing.LabelEncoder().fit_transform(X_class[col])

In [113]:
for col in categorical_columns:
    X_reg[col] = preprocessing.LabelEncoder().fit_transform(X_reg[col])

In [114]:
X_train_c, X_test_c, y_train_c, y_test_c = model_selection.train_test_split(X_class, y_class, test_size=0.2, random_state=42)

X_train_r, X_test_r, y_train_r, y_test_r = model_selection.train_test_split( X_reg, y_reg, test_size=0.2, random_state=42)

In [115]:
#Applying KNN Imputer to fill the missing values in the dataset

KNN_impute = impute.KNNImputer(n_neighbors=5)

X_class[numeric_columns] = KNN_impute.fit_transform(
    X_class[numeric_columns]
)

### Part C: Bagging (Bootstrap Aggregating)

In [116]:
bagging_classifier = ensemble.BaggingClassifier(estimator=tree.DecisionTreeClassifier(), n_estimators=100, random_state=42 )
bagging_classifier.fit(X_train_c, y_train_c)
y_pred_bagging = bagging_classifier.predict(X_test_c)
print("Bagging Classifier Accuracy:", metrics.accuracy_score(y_test_c, y_pred_bagging))

Bagging Classifier Accuracy: 0.7009615384615384


In [117]:
bagging_regressor = ensemble.BaggingRegressor(estimator=tree.DecisionTreeRegressor(), n_estimators=100, random_state=42)
bagging_regressor.fit(X_train_r, y_train_r)
y_pred_bagging_reg = bagging_regressor.predict(X_test_r)

print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred_bagging_reg))
print("RMSE:", metrics.root_mean_squared_error(y_test_r, y_pred_bagging_reg))
print("R2 Score:", metrics.r2_score(y_test_r, y_pred_bagging_reg))

MAE: 7.985228846153845
RMSE: 9.985704109645178
R2 Score: 0.4663106428917555


In [118]:
single_tree = tree.DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train_c, y_train_c)
y_pred_single = single_tree.predict(X_test_c)

print("Single Decision Tree Accuracy:", metrics.accuracy_score(y_test_c, y_pred_single))
print("Bagging Accuracy:", metrics.accuracy_score(y_test_c, y_pred_bagging))

Single Decision Tree Accuracy: 0.6240384615384615
Bagging Accuracy: 0.7009615384615384


### Part D: Boosting Algorithms

In [128]:
X_train_c=X_train_c.fillna(0)
X_train_c.isnull().sum()

X_test_c=X_test_c.fillna(0)
X_test_c.isnull().sum()

X_train_r=X_train_r.fillna(0)
X_train_r.isnull().sum()

X_test_r=X_test_r.fillna(0)
X_test_r.isnull().sum()

student_id               0
age                      0
country_region           0
device_type              0
education_background     0
course_level             0
course_category          0
course_start_date        0
week_of_year             0
sessions                 0
time_spent_hours         0
videos_watched           0
quiz_attempts            0
assignments_submitted    0
forum_posts              0
avg_quiz_score           0
attendance_rate          0
dtype: int64

AdaBoost 

In [126]:
ada_classifier = ensemble.AdaBoostClassifier( n_estimators=100, learning_rate=0.1, random_state=42)
ada_classifier.fit(X_train_c, y_train_c)
y_pred_ada = ada_classifier.predict(X_test_c)
print("AdaBoost Accuracy:", metrics.accuracy_score(y_test_c, y_pred_ada))

AdaBoost Accuracy: 0.7240384615384615


In [130]:
ada_regressor = ensemble.AdaBoostRegressor( n_estimators=100, learning_rate=0.1, random_state=42)

ada_regressor.fit(X_train_r, y_train_r)
y_pred_ada_reg = ada_regressor.predict(X_test_r)

print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred_ada_reg))
print("RMSE:", metrics.root_mean_squared_error(y_test_r, y_pred_ada_reg))
print("R2 Score:", metrics.r2_score(y_test_r, y_pred_ada_reg))

MAE: 8.515401681555558
RMSE: 10.543532759851146
R2 Score: 0.40501850393732597


Gradient Boosting

In [131]:
gradient_classifier = ensemble.GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,random_state=42 )

gradient_classifier.fit(X_train_c, y_train_c)
y_pred_gradient = gradient_classifier.predict(X_test_c)
print("Gradient Boosting Accuracy:", metrics.accuracy_score(y_test_c, y_pred_gradient))

Gradient Boosting Accuracy: 0.7201923076923077


In [133]:
gradient_regressor = ensemble.GradientBoostingRegressor( n_estimators=100, learning_rate=0.1, random_state=42 )

gradient_regressor.fit(X_train_r, y_train_r)
y_pred_gradient_reg = gradient_regressor.predict(X_test_r)

print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred_gradient_reg))
print("RMSE:", metrics.root_mean_squared_error(y_test_r, y_pred_gradient_reg))
print("R2 Score:", metrics.r2_score(y_test_r, y_pred_gradient_reg))

MAE: 7.857776746208125
RMSE: 9.842765265395036
R2 Score: 0.4814801200705975


LightGBM

In [135]:
lgb_classifier = LGBMClassifier( n_estimators=100, learning_rate=0.1, random_state=42)

lgb_classifier.fit(X_train_c, y_train_c)
y_pred_lgb = lgb_classifier.predict(X_test_c)

print("LightGBM Accuracy:", metrics.accuracy_score(y_test_c, y_pred_lgb))

LightGBM Accuracy: 0.7115384615384616


In [139]:
lgb_regressor = LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)

lgb_regressor.fit(X_train_r, y_train_r)
y_pred_lgb_reg = lgb_regressor.predict(X_test_r)

print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred_lgb_reg))
print("RMSE:", metrics.root_mean_squared_error(y_test_r, y_pred_lgb_reg))
print("R2 Score:", metrics.r2_score(y_test_r, y_pred_lgb_reg))

MAE: 8.131273903303663
RMSE: 10.125515552877232
R2 Score: 0.45126148203732397


In [140]:
print("LightGBM Accuracy:", metrics.accuracy_score(y_test_c, y_pred_lgb))

print("LightGBM Regression R2:", metrics.r2_score(y_test_r, y_pred_lgb_reg))

LightGBM Accuracy: 0.7115384615384616
LightGBM Regression R2: 0.45126148203732397


XGBoost

In [143]:
xgb_classifier = XGBClassifier( n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)

xgb_classifier.fit(X_train_c, y_train_c)
y_pred_xgb = xgb_classifier.predict(X_test_c)
print("XGBoost Accuracy:", metrics.accuracy_score(y_test_c, y_pred_xgb))

XGBoost Accuracy: 0.7163461538461539


In [145]:
xgb_regressor = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)

xgb_regressor.fit(X_train_r, y_train_r)
y_pred_xgb_reg = xgb_regressor.predict(X_test_r)
print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred_xgb_reg))
print("RMSE:", metrics.root_mean_squared_error(y_test_r, y_pred_xgb_reg))
print("R2 Score:", metrics.r2_score(y_test_r, y_pred_xgb_reg))

MAE: 7.9459621803577125
RMSE: 9.914663376968436
R2 Score: 0.47387722386504594


In [147]:
models = {
    "AdaBoost": ada_classifier,
    "Gradient Boosting": gradient_classifier,
    "LightGBM": lgb_classifier,
    "XGBoost": xgb_classifier
}

for name, model in models.items():
    y_pred = model.predict(X_test_c)
    
    print(name)
    print("Accuracy:", metrics.accuracy_score(y_test_c, y_pred))
    print()

AdaBoost
Accuracy: 0.7240384615384615

Gradient Boosting
Accuracy: 0.7201923076923077

LightGBM
Accuracy: 0.7115384615384616

XGBoost
Accuracy: 0.7163461538461539



### Part E: Voting & Stacking Ensembles

Voting

In [152]:
lr = linear_model.LogisticRegression(max_iter=1000)
knn = neighbors.KNeighborsClassifier()
svc = svm.SVC(probability=True)

voting_classifier = ensemble.VotingClassifier(
    estimators=[
        ("lr", lr),
        ("knn", knn),
        ("svc", svc)
    ],
    voting="hard"
)

voting_classifier.fit(X_train_c, y_train_c)
y_pred_voting = voting_classifier.predict(X_test_c)
print("Voting Classifier Accuracy:", metrics.accuracy_score(y_test_c, y_pred_voting))

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Voting Classifier Accuracy: 0.6567307692307692


In [153]:
hard_voting = ensemble.VotingClassifier(
    estimators=[
        ("lr", lr),
        ("knn", knn),
        ("svc", svc)
    ],
    voting="hard"
)

soft_voting = ensemble.VotingClassifier(
    estimators=[
        ("lr", lr),
        ("knn", knn),
        ("svc", svc)
    ],
    voting="soft"
)

hard_voting.fit(X_train_c, y_train_c)
soft_voting.fit(X_train_c, y_train_c)

y_pred_hard = hard_voting.predict(X_test_c)
y_pred_soft = soft_voting.predict(X_test_c)

print("Hard Voting Accuracy:", metrics.accuracy_score(y_test_c, y_pred_hard))
print("Soft Voting Accuracy:", metrics.accuracy_score(y_test_c, y_pred_soft))

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

Hard Voting Accuracy: 0.6567307692307692
Soft Voting Accuracy: 0.6509615384615385


Stacking

In [ ]:
stacking_classifier = ensemble.StackingClassifier(
    estimators=[
        ("lr", lr),
        ("knn", knn),
        ("svc", svc)
    ],
    final_estimator=linear_model.LogisticRegression(max_iter=1000)
)

stacking_classifier.fit(X_train_c, y_train_c)
y_pred_stack = stacking_classifier.predict(X_test_c)
print("Stacking Accuracy:", metrics.accuracy_score(y_test_c, y_pred_stack))

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://s

Stacking Accuracy: 0.7009615384615384


In [155]:
rf_reg = ensemble.RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

gb_reg = ensemble.GradientBoostingRegressor(
    n_estimators=100,
    random_state=42
)

stacking_regressor = ensemble.StackingRegressor(
    estimators=[
        ("rf", rf_reg),
        ("gb", gb_reg)
    ],
    final_estimator=linear_model.LinearRegression()
)

stacking_regressor.fit(X_train_r, y_train_r)
y_pred_stack_reg = stacking_regressor.predict(X_test_r)

print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred_stack_reg))
print("RMSE:", metrics.root_mean_squared_error(y_test_r, y_pred_stack_reg))
print("R2 Score:", metrics.r2_score(y_test_r, y_pred_stack_reg))

MAE: 7.851376881551121
RMSE: 9.839173081141654
R2 Score: 0.4818585257340052


### Part F: Model Evaluation & Comparison

Classification Model Evaluation

In [156]:
models = {
    "Bagging": bagging_classifier,
    "AdaBoost": ada_classifier,
    "Gradient Boosting": gradient_classifier,
    "LightGBM": lgb_classifier,
    "XGBoost": xgb_classifier,
    "Voting": soft_voting,
    "Stacking": stacking_classifier
}

for name, model in models.items():

    y_pred = model.predict(X_test_c)
    y_prob = model.predict_proba(X_test_c)[:, 1]

    print("\n", name)
    print("Accuracy:", metrics.accuracy_score(y_test_c, y_pred))
    print("Precision:", metrics.precision_score(y_test_c, y_pred))
    print("Recall:", metrics.recall_score(y_test_c, y_pred))
    print("F1 Score:", metrics.f1_score(y_test_c, y_pred))
    print("ROC-AUC:", metrics.roc_auc_score(y_test_c, y_prob))


 Bagging
Accuracy: 0.7009615384615384
Precision: 0.6357615894039735
Recall: 0.48854961832061067
F1 Score: 0.5525179856115108
ROC-AUC: 0.7433407663477156

 AdaBoost
Accuracy: 0.7240384615384615
Precision: 0.7086614173228346
Recall: 0.4580152671755725
F1 Score: 0.5564142194744977
ROC-AUC: 0.768616948059354

 Gradient Boosting
Accuracy: 0.7201923076923077
Precision: 0.6545454545454545
Recall: 0.549618320610687
F1 Score: 0.5975103734439834
ROC-AUC: 0.7717238694149154

 LightGBM
Accuracy: 0.7115384615384616
Precision: 0.6413373860182371
Recall: 0.5368956743002544
F1 Score: 0.5844875346260388
ROC-AUC: 0.7566258047516233

 XGBoost
Accuracy: 0.7163461538461539
Precision: 0.6458333333333334
Recall: 0.5521628498727735
F1 Score: 0.5953360768175583
ROC-AUC: 0.7587180606518242

 Voting
Accuracy: 0.6509615384615385
Precision: 0.5462962962962963
Recall: 0.45038167938931295
F1 Score: 0.49372384937238495
ROC-AUC: 0.6859925040606282

 Stacking
Accuracy: 0.7009615384615384
Precision: 0.6464285714285715


Regression Model Evaluation

In [157]:
models_reg = {
    "Bagging": bagging_regressor,
    "AdaBoost": ada_regressor,
    "Gradient Boosting": gradient_regressor,
    "LightGBM": lgb_regressor,
    "XGBoost": xgb_regressor,
    "Stacking": stacking_regressor
}

for name, model in models_reg.items():

    y_pred = model.predict(X_test_r)

    print("\n", name)
    print("MAE:", metrics.mean_absolute_error(y_test_r, y_pred))
    print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test_r, y_pred)))
    print("R2 Score:", metrics.r2_score(y_test_r, y_pred))


 Bagging
MAE: 8.1718625
RMSE: 10.300014997608573
R2 Score: 0.43218498843365505

 AdaBoost
MAE: 8.515401681555558
RMSE: 10.543532759851146
R2 Score: 0.40501850393732597

 Gradient Boosting
MAE: 7.857776746208125
RMSE: 9.842765265395036
R2 Score: 0.4814801200705975

 LightGBM
MAE: 8.131273903303663
RMSE: 10.125515552877232
R2 Score: 0.45126148203732397

 XGBoost
MAE: 7.9459621803577125
RMSE: 9.914663376968436
R2 Score: 0.47387722386504594

 Stacking
MAE: 7.851376881551121
RMSE: 9.839173081141654
R2 Score: 0.4818585257340052


Compare Classification Models

In [160]:
classification_results = []

for name, model in models.items():
    y_pred = model.predict(X_test_c)
    
    classification_results.append([
        name,
        metrics.accuracy_score(y_test_c, y_pred),
        metrics.precision_score(y_test_c, y_pred),
        metrics.recall_score(y_test_c, y_pred),
        metrics.f1_score(y_test_c, y_pred)
    ])

classification_results = pd.DataFrame(
    classification_results,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"]
)

print(classification_results)

               Model  Accuracy  Precision    Recall  F1 Score
0            Bagging  0.700962   0.635762  0.488550  0.552518
1           AdaBoost  0.724038   0.708661  0.458015  0.556414
2  Gradient Boosting  0.720192   0.654545  0.549618  0.597510
3           LightGBM  0.711538   0.641337  0.536896  0.584488
4            XGBoost  0.716346   0.645833  0.552163  0.595336
5             Voting  0.650962   0.546296  0.450382  0.493724
6           Stacking  0.700962   0.646429  0.460560  0.537890


Compare Regression Models

In [164]:
regression_results = []

for name, model in models_reg.items():
    y_pred = model.predict(X_test_r)
    
    regression_results.append([
        name,
        metrics.mean_absolute_error(y_test_r, y_pred),
        metrics.root_mean_squared_error(y_test_r, y_pred),
        metrics.r2_score(y_test_r, y_pred)
    ])

regression_results = pd.DataFrame(
    regression_results,
    columns=["Model", "MAE", "RMSE", "R2 Score"]
)

print(regression_results)

               Model       MAE       RMSE  R2 Score
0            Bagging  8.171862  10.300015  0.432185
1           AdaBoost  8.515402  10.543533  0.405019
2  Gradient Boosting  7.857777   9.842765  0.481480
3           LightGBM  8.131274  10.125516  0.451261
4            XGBoost  7.945962   9.914663  0.473877
5           Stacking  7.851377   9.839173  0.481859
